In [57]:
import langchain
from extract4chem_fluorine.tool import processDoc
from pathlib import Path
import os

In [58]:
markdown_text = Path("../data/raw/full.md").read_text(encoding="utf-8")

In [59]:
splited_texts = processDoc(markdown_text)

# 准备数据

In [60]:
start_index = 0
end_index = len(splited_texts)
end_flags = ["acknowledgement", "references", "appendix"]
for i, t in enumerate(splited_texts):
    if "abstract" in t["content_title"].lower():
        start_index = i
    elif any(flag in t["content_title"].lower() for flag in end_flags) and i > start_index:
        end_index = i
        break
eff_texts = splited_texts[start_index:end_index]

In [61]:
eff_paper = "\n\n".join([f"# {t["content_title"]}\n{t["content"]}" for t in eff_texts])

In [62]:
eff_paper

'# ABSTRACT\nThe conversion of methanol-to-hydrocarbons (MTH) has been studied over a series of  $\\mathrm{Zn / ZSM - 5}$  zeolites with different thickness of b-axis, as well as similar lengths of a-axis. It has been demonstrated that the decrease of b-axis thickness from  $220\\mathrm{nm}$  to  $2\\mathrm{nm}$  leads to the remarkably longer lifetime, accompanied by the shift of selectivity toward trimethylbenzene and increased coke tolerance capacity. Methylbenzenes, as the intermediate product of the aromatic-based cycle, can diffuse out of the straight channels in the  $\\mathrm{Zn / ZSM - 5}$  nanosheet quickly, suppressing the aromatic-based cycle. The evolution of coke species, including the quantity, types, and location, as a function of the reaction time, has been systematically investigated. During the initial reaction period, the coke preferentially forms in mesopores and then is deposited mainly in micropores as the reaction proceeds. The  $\\mathrm{Zn / ZSM - 5}$  nanoshe

# 准备模型

In [75]:
model_config = {
    "model": "doubao-seed-1-6-thinking-250715",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": os.getenv("DOUBAO_BASEURL"),        # 这里替换为自己的 Base URL
    "api_key": os.getenv("DOUBAO_APIKEY_FS"),       # 这里替换为自己的 Key
    "max_completion_tokens" : 64000
}

In [64]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(**model_config)

In [67]:
# 模型验证
llm.invoke("你是谁")

AIMessage(content='我是由字节跳动公司开发的人工智能，名叫豆包，致力于为用户提供各类信息查询、交流互动等服务～', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 11, 'total_tokens': 142, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 104, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-6-thinking-250715', 'system_fingerprint': None, 'id': '021761282295417f65904c546c85e120f302573011ada0238099e', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--11b6c5c2-06f4-4967-9be5-7e11edd1a4f7-0', usage_metadata={'input_tokens': 11, 'output_tokens': 131, 'total_tokens': 142, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 104}})

In [69]:
from langchain_core.prompts import ChatPromptTemplate
prompt_temp = ChatPromptTemplate.from_template(Path("../prompts/抽取.txt").read_text(encoding="utf-8"))
prompt_temp.pretty_print()

================================ Human Message =================================

<System>
请阅读提供的中文或英文文献片段，识别其中与含氟抽取材料相关的结构化信息。输出时遵循以下通用原则：
- 所有结论必须有原文依据，不得杜撰；若信息缺失，输出字段值为 `null`。
- 数值统一使用文献中给定的单位；若需换算，请在说明中写出换算逻辑并给出换算后的数值。
- 若文献给出了质量或体积等原始数据，需要据此计算出目标摩尔配比后再填入字段。
- 不允许遗漏字段。即便无法获得信息，也要显式写出对应键并赋值为 `null`。

输出要求：
- 返回合法的 JSON 字符串，可被严格模式解析。
- 顶层键：`material_id`、`doi`、`synthesis`、`characterization`、`application`。
- 任何列表采用明确顺序。若无数据，使用 `null` 而非空列表。
- 用英文输出

字段约束说明（采用点号表示层级）：
- `material_id`：材料在文献中的命名，可含字母、数字及符号。
- `doi`：文献 DOI。若正文未出现，填入 `null`。
- `synthesis.zeolite_type`：框架结构代码（如 MFI、BEA）。
- `synthesis.gel_composition`：凝胶摩尔比，字段顺序应与文献一致。
- `synthesis.template`：结构导向剂完整名称或缩写。
- `synthesis.silica_source` / `synthesis.aluminium_source`：原料名称。
- `synthesis.crystallization.temp_c` / `time_d` / `agitation_rpm`：晶化条件，缺失时写 `null`。
- `synthesis.post_treatment_steps`：后处理步骤，按文献顺序，用“操作_补充说明”命名。
- `characterization.morphology`：形貌关键词（如 sheet、flower-like）。
- `characterization.crystal_size.{a_axis_nm,b_axis_nm,c

In [70]:
from extract4chem_fluorine.entities.抽取 import ExtractionResults
ExtractionResults.model_json_schema()

{'$defs': {'Acidity': {'additionalProperties': False,
   'properties': {'bronsted_acid_amount_mmol_g': {'anyOf': [{'type': 'number'},
      {'type': 'null'}],
     'default': None,
     'description': 'Brønsted 酸含量（mmol/g）',
     'title': 'Bronsted Acid Amount Mmol G'},
    'lewis_acid_amount_mmol_g': {'anyOf': [{'type': 'number'},
      {'type': 'null'}],
     'default': None,
     'description': 'Lewis 酸含量（mmol/g）',
     'title': 'Lewis Acid Amount Mmol G'},
    'l_b_ratio': {'anyOf': [{'type': 'number'}, {'type': 'null'}],
     'default': None,
     'description': 'Lewis/Brønsted 酸含量比值',
     'title': 'L B Ratio'}},
   'title': 'Acidity',
   'type': 'object'},
  'Application': {'additionalProperties': False,
   'properties': {'scenario': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'description': '应用场景',
     'title': 'Scenario'},
    'target_species': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'description': '反应底物或目标

In [71]:
chain = prompt_temp | llm.with_structured_output(ExtractionResults)

In [72]:
result = chain.invoke({"paper": eff_paper})

# 结果序列化

In [73]:
outpath = Path("../data/out/extraction_results.json")

In [74]:
import orjsonl
orjsonl.save(outpath, [result.model_dump(mode = "json")])